In [23]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone

import os
import sys
sys.path.append(os.path.abspath('./src'))

from db_functions import DotaDB

db = DotaDB()

In [20]:
df['dire_team_id'].isna().sum()

np.int64(135159)

In [24]:
## Calculate form: for now, wins out of last 5 games, simple int
query = 'SELECT * FROM match_details'
df = pd.DataFrame(db.query_select_to_df(query, table='match_details'))
df = df.sort_values(by='startDateTime', axis=0, ascending=False)
rad = df[['id', 'radiantTeamId', 'didRadiantWin', 'startDateTime']].rename(
    columns={'radiantTeamId': 'teamId', 'didRadiantWin': 'won'}
)
dire = df[['id', 'direTeamId', 'didRadiantWin', 'startDateTime']].rename(
    columns={'direTeamId': 'teamId', 'didRadiantWin': 'won'}
)
dire['won'] = ~dire['won'] 

team_history = pd.concat([rad, dire]).sort_values(['teamId', 'startDateTime'])

team_history['form'] = (
    team_history.groupby('teamId')['won']
    .apply(lambda x: x.shift(1).rolling(window=5, min_periods=1).sum())
    .reset_index(level=0, drop=True)
)

df = df.merge(
    team_history[['id', 'teamId', 'form']], 
    left_on=['id', 'radiantTeamId'], 
    right_on=['id', 'teamId'], 
    how='left'
).rename(columns={'form': 'radiantForm'}).drop('teamId', axis=1)

df = df.merge(
    team_history[['id', 'teamId', 'form']], 
    left_on=['id', 'direTeamId'], 
    right_on=['id', 'teamId'], 
    how='left'
).rename(columns={'form': 'direForm'}).drop('teamId', axis=1)

In [25]:
df

,id,tournamentId,tournamentRound,leagueId,radiantTeamId,direTeamId,seriesId,gameVersionId,regionId,clusterId,...,topLaneOutcome,midLaneOutcome,bottomLaneOutcome,predictedOutcomeWeight,startDateTimeHuman,endDateTimeHuman,avg_radiant_rating,avg_dire_rating,radiantForm,direForm
0,8702535685,None,None,19201,9048057,8849990,1067737,182,3,272,...,TIE,TIE,DIRE_VICTORY,53.0,2026-02-23 00:46:32,2026-02-23 01:27:13,262.495552,258.533818,3.0,0.0
1,8702524545,None,None,19218,8969258,8835624,1067717,182,15,251,...,TIE,TIE,RADIANT_VICTORY,52.0,2026-02-23 00:26:23,2026-02-23 01:14:00,18.006482,19.610925,3.0,3.0
2,8702512244,None,None,19201,8987220,8850015,1067719,182,3,272,...,RADIANT_STOMP,RADIANT_VICTORY,DIRE_VICTORY,66.0,2026-02-23 00:06:12,2026-02-23 00:35:23,262.443691,261.192097,3.0,2.0
3,8702504924,None,None,16251,9823250,9361469,1067727,182,8,183,...,TIE,RADIANT_VICTORY,RADIANT_VICTORY,NaN,2026-02-22 23:54:03,2026-02-23 00:20:31,10.466518,25.162645,1.0,3.0
4,8702486501,None,None,19201,8850015,8987220,1067719,182,3,274,...,RADIANT_VICTORY,TIE,DIRE_VICTORY,60.0,2026-02-22 23:27:37,2026-02-22 23:56:27,261.473490,261.775087,2.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120017,6321928009,None,None,13690,8449479,8546012,626075,149,3,273,...,None,None,None,72.0,2021-12-15 16:05:51,2021-12-15 16:50:10,107.140698,101.377400,NaN,NaN
120018,6321818813,None,None,13379,8261774,8588969,626082,149,3,274,...,None,None,None,43.0,2021-12-15 15:21:15,2021-12-15 15:55:09,154.587936,150.002979,NaN,NaN
120019,6321807530,None,None,13762,8599633,7653080,626065,149,5,156,...,None,None,None,60.0,2021-12-15 15:10:43,2021-12-15 15:39:50,64.250175,130.671000,NaN,NaN
120020,6321792491,None,None,13738,8291895,2163,626062,149,3,273,...,None,None,None,53.0,2021-12-15 15:08:04,2021-12-15 15:45:23,164.840048,170.204678,NaN,NaN


In [26]:
## Roster longevity
##TODO: hero longevity and othe ideas
query = '''
    SELECT match_id, mp."heroId", mp.position, mp."isRadiant", mp."isVictory",
        variant, mp."steamAccountId", name, mp."realName",
        md."radiantTeamId", md."direTeamId", md."startDateTime"
    FROM match_players mp
    INNER JOIN match_details md
    ON md.id = mp.match_id
'''
results = db.query_select(query) 
df_roster = pd.DataFrame(
    results, 
    columns=[
        'matchId', 
        'heroId', 
        'position',
        'isRadiant', 
        'isVictory', 
        'variant', 
        'steamAccountId', 
        'name', 
        'realName', 
        'radiantTeamId', 
        'direTeamId', 
        'startDateTime'
    ]
)
df_roster['realName'].value_counts() ## Most of them don't have any
df_roster = df_roster.drop('realName', axis=1).sort_values(by='startDateTime', ascending=False)
roster_groups = (
    df_roster.sort_values(['matchId', 'isRadiant', 'steamAccountId'])
    .groupby(['matchId', 'isRadiant'])['steamAccountId']
    .apply(tuple)
    .reset_index()
)

roster_groups.columns = ['matchId', 'isRadiant', 'roster_tuple']

df = df.merge(roster_groups[roster_groups['isRadiant'] == True], left_on='id', right_on='matchId', how='left')
df = df.rename(columns={'roster_tuple': 'rad_roster_tuple'}).drop('isRadiant', axis=1)

df = df.merge(roster_groups[roster_groups['isRadiant'] == False], left_on='id', right_on='matchId', how='left')
df = df.rename(columns={'roster_tuple': 'dire_roster_tuple'}).drop('isRadiant', axis=1)

rad_rosters = df[['id', 'radiantTeamId', 'rad_roster_tuple', 'startDateTime']].rename(
    columns={'radiantTeamId': 'teamId', 'rad_roster_tuple': 'roster'}
)
dire_rosters = df[['id', 'direTeamId', 'dire_roster_tuple', 'startDateTime']].rename(
    columns={'direTeamId': 'teamId', 'dire_roster_tuple': 'roster'}
)

roster_history = pd.concat([rad_rosters, dire_rosters]).sort_values(['teamId', 'roster', 'startDateTime'])
roster_history['roster_experience'] = roster_history.groupby(['teamId', 'roster']).cumcount()

df = df.merge(
    roster_history[['id', 'teamId', 'roster_experience']], 
    left_on=['id', 'radiantTeamId'], 
    right_on=['id', 'teamId'], 
    how='left'
).rename(columns={'roster_experience': 'rad_roster_exp'}).drop('teamId', axis=1)

# Merge for Dire
df = df.merge(
    roster_history[['id', 'teamId', 'roster_experience']], 
    left_on=['id', 'direTeamId'], 
    right_on=['id', 'teamId'], 
    how='left'
).rename(columns={'roster_experience': 'dire_roster_exp'}).drop('teamId', axis=1)

In [31]:
## Calculating lane longevity
df_roster = df_roster.sort_values(['startDateTime', 'matchId'])
lanes = []
for (match_id, is_radiant), group in df_roster.groupby(['matchId', 'isRadiant']):
    pos_map = group.set_index('position')['steamAccountId'].to_dict()
    if 'POSITION_1' in pos_map and 'POSITION_5' in pos_map:
        lanes.append({
            'matchId': match_id,
            'pair': tuple(sorted([pos_map['POSITION_1'], pos_map['POSITION_5']])),
            'lane_type': 'Safelane',
            'timestamp': group['startDateTime'].iloc[0],
            'isRadiant': is_radiant
        })
        
    # Offlane Duo (Pos 3 & 4)
    if 'POSITION_3' in pos_map and 'POSITION_4' in pos_map:
        lanes.append({
            'matchId': match_id,
            'pair': tuple(sorted([pos_map['POSITION_3'], pos_map['POSITION_4']])),
            'lane_type': 'Offlane',
            'timestamp': group['startDateTime'].iloc[0],
            'isRadiant': is_radiant
        })

lane_df = pd.DataFrame(lanes)
lane_df['games_together_count'] = lane_df.groupby('pair').cumcount()
rad_safelane_long = lane_df[(lane_df['lane_type'] == 'Safelane') & (lane_df['isRadiant'] == True)].rename(
    {'games_together_count': 'radiant_safelane_long'}, axis=1
)
rad_offlane_long = lane_df[(lane_df['lane_type'] == 'Offlane') & (lane_df['isRadiant'] == True)].rename(
    {'games_together_count': 'radiant_offlane_long'}, axis=1
)
dire_safelane_long = lane_df[(lane_df['lane_type'] == 'Safelane') & (lane_df['isRadiant'] == False)].rename(
    {'games_together_count': 'dire_safelane_long'}, axis=1
)
dire_offlane_long = lane_df[(lane_df['lane_type'] == 'Offlane') & (lane_df['isRadiant'] == False)].rename(
    {'games_together_count': 'dire_offlane_long'}, axis=1
)

df = pd.concat(
    [
        df.reset_index(), 
        rad_safelane_long.iloc[:, -1].reset_index(), 
        rad_offlane_long.iloc[:, -1].reset_index(), 
        dire_safelane_long.iloc[:, -1].reset_index(), 
        dire_offlane_long.iloc[:, -1].reset_index()
    ], 
    axis=1
)

In [45]:
## Calculating draft strength
# df_roster['hero_composite'] = df_roster['heroId'].astype(str) + '_' + df_roster['variant'].astype(str)
query = '''
        SELECT 
        mp."heroId" as hero_id,
        md."gameVersionId",
        AVG(CAST(mp."isVictory" AS INT)) as winrate,
        COUNT(*)
    FROM match_players mp
    JOIN match_details md ON md.id = mp.match_id
    GROUP BY mp."heroId", md."gameVersionId"
    HAVING COUNT(*) >= 20
    '''
hero_winrates = db.query_select_to_df(query, columns=['hero_id', 'patch', 'winrate', 'match_count'])
query = '''
    SELECT 
        LEAST(mp1."heroId", mp2."heroId")    as hero1,
        GREATEST(mp1."heroId", mp2."heroId") as hero2,
        AVG(CAST(mp1."isVictory" AS INT))    as pair_winrate,
        COUNT(*) 
    FROM match_players mp1
    JOIN match_players mp2 
        ON mp1.match_id = mp2.match_id
        AND mp1."heroId" < mp2."heroId"
        AND mp1."isRadiant" = mp2."isRadiant"
    GROUP BY 1, 2
    HAVING COUNT(*) >= 15
    ORDER BY pair_winrate DESC
    '''
hero_synergy = db.query_select_to_df(query, columns=['hero_id1', 'hero_id2', 'pair_winrate', 'match_count'])
query = '''
    SELECT 
        mp1."heroId" as hero_id,
        mp2."heroId" as enemy_hero_id,
        AVG(CAST(mp1."isVictory" AS INT)) as winrate_vs,
        COUNT(*) as games
    FROM match_players mp1
    JOIN match_players mp2
        ON mp1.match_id = mp2.match_id
        AND mp1."isRadiant" != mp2."isRadiant"
    GROUP BY 1, 2
    HAVING COUNT(*) >= 15
    '''
hero_counters = db.query_select_to_df(query, columns=['hero_id', 'enemy_hero_id', 'winrate_vs', 'match_count'])

In [ ]:
def draft_strength(team_heroes, enemy_heroes, patch,
                   hero_winrates, hero_synergy, hero_counters):
    """
    Compute a draft strength score for a team given their heroes and the enemy heroes.
    Returns a float — higher is stronger draft.

    team_heroes:  list of hero_ids for this team
    enemy_heroes: list of hero_ids for the enemy team
    patch:        current patch int
    """
    # ── 1. Individual hero win rates ─────────────────────────────────────────
    wr_scores = []
    for hero_id in team_heroes:
        # try patch-specific first, fall back to overall
        patch_wr = hero_winrates[
            (hero_winrates['hero_id'] == hero_id) &
            (hero_winrates['patch'] == patch) 
        ]['winrate']

        if len(patch_wr) > 0:
            wr_scores.append(patch_wr.iloc[0])
        else:
            overall_wr = hero_winrates[
                (hero_winrates['hero_id'] == hero_id)
            ]['winrate']
            if len(overall_wr) > 0:
                wr_scores.append(overall_wr.mean())
            else:
                wr_scores.append(0.50)  # unknown hero — assume neutral

    hero_wr_score = np.mean(wr_scores)

    # ── 2. Team synergy — all hero pairs ────────────────────────────────────
    synergy_scores = []
    for i, h1 in enumerate(team_heroes):
        for h2 in team_heroes[i+1:]:
            key_h1 = min(h1, h2)
            key_h2 = max(h1, h2)
            pair = hero_synergy[
                (hero_synergy['hero1'] == key_h1) &
                (hero_synergy['hero2'] == key_h2) 
            ]['pair_winrate']
            if len(pair) > 0:
                synergy_scores.append(pair.iloc[0])

    synergy_score = np.mean(synergy_scores) if synergy_scores else 0.50

    # ── 3. Counter score — how well this team counters the enemy ────────────
    counter_scores = []
    for hero_id in team_heroes:
        for enemy_id in enemy_heroes:
            matchup = hero_counters[
                (hero_counters['hero_id'] == hero_id) &
                (hero_counters['enemy_hero_id'] == enemy_id) 
            ]['winrate_vs']
            if len(matchup) > 0:
                counter_scores.append(matchup.iloc[0])

    counter_score = np.mean(counter_scores) if counter_scores else 0.50

    # ── Weighted combination ─────────────────────────────────────────────────
    draft_score = (
        hero_wr_score  * 0.40 +
        synergy_score  * 0.35 +
        counter_score  * 0.25
    )
    return draft_score